# Enhanced XAI-IDS Training Pipeline



In [ ]:
!pip -q install lightgbm shap scikit-learn pandas numpy matplotlib seaborn joblib xgboost catboost lime optuna torch
from google.colab import drive
drive.mount('/content/drive')

## 1. Imports & Configuration

In [ ]:
import os
import glob
import json
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import shap
import joblib

# FT-Transformer imports
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Optional: LIME for cross-validation
try:
    import lime
    import lime.lime_tabular
    LIME_AVAILABLE = True
except ImportError:
    LIME_AVAILABLE = False
    print("LIME not installed. LIME cross-validation will be skipped.")

RANDOM_STATE = 42
RANDOM_SEEDS = [42, 123, 456]          # Multi-seed evaluation
TOP_K_SIGNATURE_FEATURES = 15
MIN_ROWS_PER_CLASS_FOR_SIGNATURE = 10
MAX_SHAP_ROWS = 3000

# NOT used for final training or final results.
RUN_FEATURE_SELECTION_CONSISTENCY_DIAGNOSTIC = True
FEATURE_SELECTION_TOP_N_DIAGNOSTIC = 20

# Split settings
MIN_CLASS_COUNT = 30
MIN_LABELS_PER_SPLIT = 4
SEEN_HOST_RATIO = 0.7
MAX_SPLIT_TRIES = 30

# decision Policy constraints
MIN_ATTACK_DETECTION_RATE = 0.95       # Minimum acceptable attack detection

# Benign filtering requested by reviewer fix
KEEP_BENIGN_FILES = {"friday_benign.csv", "thursday_benign.csv"}

# Ports removed as possible leakage features
PORT_COLUMNS_TO_DROP = ["src_port", "dst_port"]

# Paths
DATA_FOLDER = "/content/drive/MyDrive/BCCC-CIC-IDS-2017"
DRIVE_RESULTS_PATH = "/content/drive/MyDrive/IDS_XAI_Project/results_main2"

print("=== Configuration ===")
for k, v in {
    "RANDOM_SEEDS": RANDOM_SEEDS,
    "MAX_SHAP_ROWS": MAX_SHAP_ROWS,
    "MIN_CLASS_COUNT": MIN_CLASS_COUNT,
    "SEEN_HOST_RATIO": SEEN_HOST_RATIO,
    "KEEP_BENIGN_FILES": sorted(KEEP_BENIGN_FILES),
    "PORT_COLUMNS_TO_DROP": PORT_COLUMNS_TO_DROP,
    "RUN_FEATURE_SELECTION_CONSISTENCY_DIAGNOSTIC": RUN_FEATURE_SELECTION_CONSISTENCY_DIAGNOSTIC,
}.items():
    print(f"  {k} = {v}")

In [ ]:
os.makedirs(DRIVE_RESULTS_PATH, exist_ok=True)
print("Results will be saved to:", DRIVE_RESULTS_PATH)

## 2. Data Loading & Cleaning

In [ ]:
csv_files = sorted(glob.glob(os.path.join(DATA_FOLDER, "*.csv")))
assert len(csv_files) > 0, f"No CSV files found in {DATA_FOLDER}"

print("Files found:")
for f in csv_files:
    print(" -", os.path.basename(f))

dfs = []
for f in csv_files:
    fname = os.path.basename(f)

    # To reduce class imbalance, keep only friday_benign.csv and thursday_benign.csv for benign traffic.
    # Any other benign-only file is skipped to reduce benign dominance/class imbalance.
    if "benign" in fname.lower() and fname.lower() not in KEEP_BENIGN_FILES:
        print(f"Skipped benign file: {fname}")
        continue

    df = pd.read_csv(f, low_memory=False)
    df["source_file"] = fname
    dfs.append(df)
    print(f"Loaded {fname} -> {df.shape}")

assert len(dfs) > 0, "No files loaded after benign filtering. Check DATA_FOLDER and KEEP_BENIGN_FILES."

data = pd.concat(dfs, ignore_index=True)
data.columns = [c.strip() for c in data.columns]

print("\nCombined shape after benign-file filtering:", data.shape)
print("\nFiles retained:")
print(data["source_file"].value_counts())
print("\nColumns:")
print(data.columns.tolist())

In [ ]:
required_cols = ["flow_id", "timestamp", "src_ip", "dst_ip", "label"]
for c in required_cols:
    assert c in data.columns, f"Missing required column: {c}"

data["timestamp"] = pd.to_datetime(data["timestamp"], errors="coerce")
data["label"] = data["label"].astype(str).str.strip()

data = data.dropna(subset=["timestamp", "src_ip", "label"]).reset_index(drop=True)
data = data.drop_duplicates().reset_index(drop=True)

data["Date"] = data["timestamp"].dt.date
data["HostID"] = data["src_ip"].astype(str)

print("Shape after cleaning:", data.shape)
print("Unique dates:", data["Date"].nunique())
print("Unique hosts:", data["HostID"].nunique())
print("Unique labels:", data["label"].nunique())
print("\nTop label counts:")
print(data["label"].value_counts().head(20))

## 3. Train / Validation / Test Splitting

In [ ]:

# 1. Remove tiny classes

label_counts = data["label"].value_counts()
keep_labels = label_counts[label_counts >= MIN_CLASS_COUNT].index
dropped_labels = label_counts[label_counts < MIN_CLASS_COUNT]

if len(dropped_labels) > 0:
    print("Dropping tiny classes:")
    print(dropped_labels)

data_split = data[data["label"].isin(keep_labels)].copy()

print("\nRemaining labels:")
print(data_split["label"].value_counts())



# 2. Split EACH scenario file by time (60/20/20)

train_parts, val_parts, test_parts = [], [], []

for fname, df_file in data_split.groupby("source_file"):
    df_file = df_file.sort_values("timestamp")
    n = len(df_file)
    train_end = int(0.6 * n)
    val_end = int(0.8 * n)

    train_parts.append(df_file.iloc[:train_end])
    val_parts.append(df_file.iloc[train_end:val_end])
    test_parts.append(df_file.iloc[val_end:])

train_df = pd.concat(train_parts).copy()
val_base_df = pd.concat(val_parts).copy()
test_base_df = pd.concat(test_parts).copy()

print("\nAfter scenario time split:")
print("Train rows:", len(train_df))
print("Validation rows:", len(val_base_df))
print("Test rows:", len(test_base_df))



# 3. Host split helper

def split_hosts(df, seen_ratio=0.7, random_state=42):
    hosts = np.array(sorted(df["HostID"].unique()))
    rng = np.random.default_rng(random_state)
    rng.shuffle(hosts)

    n_seen = max(1, int(len(hosts) * seen_ratio))
    seen_hosts = set(hosts[:n_seen])

    seen_df = df[df["HostID"].isin(seen_hosts)].copy()
    unseen_df = df[~df["HostID"].isin(seen_hosts)].copy()

    return seen_df, unseen_df


# 4. Evaluate multiple host split ratios

def print_split_info(name, df):
    print(f"\n{name}")
    print("Rows:", len(df))
    print("Hosts:", df["HostID"].nunique())
    print("Labels:", df["label"].nunique())
    print(df["label"].value_counts())


def evaluate_host_split_ratios(val_df_in, test_df_in, ratios=(0.7, 0.6, 0.5, 0.4), random_state=42):
    rows = []

    for ratio in ratios:
        val_seen_df, host_unseen_df = split_hosts(val_df_in, seen_ratio=ratio, random_state=random_state)
        time_seen_df, hard_unseen_df = split_hosts(test_df_in, seen_ratio=ratio, random_state=random_state)

        rows.append({
            "seen_ratio": ratio,
            "unseen_ratio": round(1 - ratio, 2),
            "val_rows": len(val_seen_df),
            "val_hosts": val_seen_df["HostID"].nunique(),
            "val_labels": val_seen_df["label"].nunique(),
            "host_rows": len(host_unseen_df),
            "host_hosts": host_unseen_df["HostID"].nunique(),
            "host_labels": host_unseen_df["label"].nunique(),
            "time_rows": len(time_seen_df),
            "time_hosts": time_seen_df["HostID"].nunique(),
            "time_labels": time_seen_df["label"].nunique(),
            "hard_rows": len(hard_unseen_df),
            "hard_hosts": hard_unseen_df["HostID"].nunique(),
            "hard_labels": hard_unseen_df["label"].nunique(),
        })

    return pd.DataFrame(rows)


ratio_results = evaluate_host_split_ratios(
    val_base_df,
    test_base_df,
    ratios=(0.7, 0.6, 0.5, 0.4),
    random_state=RANDOM_STATE
)

print("\nHost split ratio experiment summary:")
display(ratio_results)



# 5. Choose the best host split ratio
#    Priority:
#    1) keep full label coverage in VALIDATION
#    2) keep full label coverage in TIME-TEST
#    3) maximize labels in HOST-TEST
#    4) maximize labels in HARD-TEST
#    5) keep more validation rows

full_label_count = data_split["label"].nunique()

ratio_results["val_label_loss"] = full_label_count - ratio_results["val_labels"]
ratio_results["time_label_loss"] = full_label_count - ratio_results["time_labels"]

ratio_results_sorted = ratio_results.sort_values(
    by=[
        "val_label_loss",   # smaller is better
        "time_label_loss",  # smaller is better
        "host_labels",      # larger is better
        "hard_labels",      # larger is better
        "val_rows",         # larger is better
        "time_rows"         # larger is better
    ],
    ascending=[True, True, False, False, False, False]
).reset_index(drop=True)

best_seen_ratio = ratio_results_sorted.loc[0, "seen_ratio"]
best_unseen_ratio = ratio_results_sorted.loc[0, "unseen_ratio"]

print("\nSelected host split ratio:")
print(f"Seen/Unseen = {best_seen_ratio:.1f}/{best_unseen_ratio:.1f}")

print("\nSorted ratio comparison:")
display(ratio_results_sorted)

In [ ]:
# Apply selected host split ratio to validation and test
val_df, host_test_df = split_hosts(val_base_df, seen_ratio=best_seen_ratio, random_state=RANDOM_STATE)
time_test_df, hard_test_df = split_hosts(test_base_df, seen_ratio=best_seen_ratio, random_state=RANDOM_STATE)

print("Final dataset shapes:")
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Host-Test:", host_test_df.shape)
print("Time-Test:", time_test_df.shape)
print("Hard-Test:", hard_test_df.shape)

## 4. Feature Engineering & Robust Label Encoding

In [ ]:
leakage_cols = [
    "flow_id", "timestamp", "Date", "src_ip", "dst_ip",
    "HostID", "label", "source_file",
    "src_port", "dst_port"
]

numeric_feature_cols = [
    c for c in train_df.columns
    if c not in leakage_cols and pd.api.types.is_numeric_dtype(train_df[c])
]

assert len(numeric_feature_cols) > 0, "No numeric model features found."
print("Number of raw behavioral features after identifier + port removal:", len(numeric_feature_cols))
print("Dropped leakage/metadata columns:", [c for c in leakage_cols if c in train_df.columns])


def clean_X(df, feature_cols):
    X = df[feature_cols].copy()
    X = X.replace([np.inf, -np.inf], np.nan)
    return X


X_train = clean_X(train_df, numeric_feature_cols)
X_val = clean_X(val_df, numeric_feature_cols)
X_host_test = clean_X(host_test_df, numeric_feature_cols)
X_time_test = clean_X(time_test_df, numeric_feature_cols)
X_hard_test = clean_X(hard_test_df, numeric_feature_cols)

# This notebook does NOT perform SHAP feature selection for final models.

selected_features = numeric_feature_cols


missing_summary = pd.DataFrame({
    "feature": X_train.columns,
    "train_missing_rate": X_train.isna().mean().values,
    "val_missing_rate": X_val.isna().mean().values,
    "time_test_missing_rate": X_time_test.isna().mean().values,
    "hard_test_missing_rate": X_hard_test.isna().mean().values,
}).sort_values("train_missing_rate", ascending=False)

print("\nTop missing-value features:")
display(missing_summary.head(20))
missing_summary.to_csv(os.path.join(DRIVE_RESULTS_PATH, "missingness_summary.csv"), index=False)

# Robust Label Encoding: handle unknown labels gracefully
le = LabelEncoder()
y_train_raw = train_df["label"].astype(str).values
le.fit(y_train_raw)

known_classes = set(le.classes_)


def safe_transform(labels, label_encoder, unknown_label="UNKNOWN"):
    """Transform labels, mapping any unseen labels to a dedicated UNKNOWN class."""
    labels = np.array(labels, dtype=str)
    known = set(label_encoder.classes_)
    unknown_mask = np.array([l not in known for l in labels])

    if unknown_mask.any():
        n_unknown = unknown_mask.sum()
        unknown_names = set(labels[unknown_mask])
        print(f"  WARNING: {n_unknown} samples with unknown labels: {unknown_names}")
        print(f"  These will be mapped to '{unknown_label}'.")

        new_classes = np.append(label_encoder.classes_, unknown_label)
        label_encoder.classes_ = new_classes
        labels[unknown_mask] = unknown_label

    return label_encoder.transform(labels)


y_train = le.transform(y_train_raw)
y_val = safe_transform(val_df["label"].astype(str).values, le)
y_host_test = safe_transform(host_test_df["label"].astype(str).values, le)
y_time_test = safe_transform(time_test_df["label"].astype(str).values, le)
y_hard_test = safe_transform(hard_test_df["label"].astype(str).values, le)

print("\nClasses:", le.classes_)

benign_candidates = [i for i, c in enumerate(le.classes_) if "benign" in c.lower()]
benign_idx = benign_candidates[0] if benign_candidates else None
print("\nBenign index:", benign_idx)
if benign_idx is not None:
    print("Benign class:", le.classes_[benign_idx])

In [ ]:
import numpy as np

# 1. Check for standard numeric missing values (what Pandas catches)
standard_nan_count = X_train.isna().sum().sum()

# 2. Check for hidden text/string matches of "NaN" or "Infinity" that Excel masks
string_nan_count = X_train.astype(str).isin(['NaN', 'nan', 'Infinity', 'inf', '-Infinity', '-inf']).sum().sum()

print(f"Standard numeric NaNs caught by Pandas: {standard_nan_count}")
print(f"Hidden text strings (NaN/Infinity) matching exactly: {string_nan_count}")

# 3. Print exactly which columns contain these values so you can name them safely
columns_with_anomalies = X_train.columns[X_train.astype(str).isin(['NaN', 'nan', 'Infinity', 'inf', '-Infinity', '-inf']).any()]
print(f"Columns containing these scale anomalies: {list(columns_with_anomalies)}")

## 5. Model Training (LightGBM, XGBoost, CatBoost, Random Forest, FT-Transformer)

In [ ]:


# =====================================================================
# STEP 5: CENTRALIZED DATA IMPUTATION & MULTI-MODEL TRAINING SPACE
# =====================================================================
# A global imputer handles missing value anomalies uniformly before fitting.

print("Executing global dataset transformation (Median Imputation Only)...")
global_imputer = SimpleImputer(strategy="median")


X_train_imp = pd.DataFrame(
    global_imputer.fit_transform(X_train),
    columns=X_train.columns
)
X_val_imp = pd.DataFrame(
    global_imputer.transform(X_val),
    columns=X_val.columns
)

n_classes = len(le.classes_)
print(f"Training {n_classes}-class models on native-scale feature space...\n")
timing = {}

# -----------------------
# 5a. LightGBM
# -----------------------
t0 = time.time()
lgb_model = lgb.LGBMClassifier(
    objective="multiclass",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=64,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    n_jobs=-1,
    verbose=-1
)
lgb_model.fit(
    X_train_imp, y_train,
    eval_set=[(X_val_imp, y_val)],
    eval_metric="multi_logloss",
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)]
)
timing["LightGBM_train"] = time.time() - t0
print(f"LightGBM trained in {timing['LightGBM_train']:.1f}s\n")

# -----------------------
# 5b. XGBoost
# -----------------------
t0 = time.time()
xgb_model = xgb.XGBClassifier(
    objective="multi:softprob",
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    random_state=RANDOM_STATE,
    use_label_encoder=False,
    eval_metric="mlogloss",
    n_jobs=-1,
    verbosity=0
)
sample_weights = compute_sample_weight("balanced", y_train)
xgb_model.fit(
    X_train_imp, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_val_imp, y_val)],
    verbose=False
)
timing["XGBoost_train"] = time.time() - t0
print(f"XGBoost trained in {timing['XGBoost_train']:.1f}s\n")

# -----------------------
# 5c. CatBoost
# -----------------------
t0 = time.time()
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
cb_model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.05,
    depth=8,
    loss_function="MultiClass",
    eval_metric="MultiClass",
    random_seed=RANDOM_STATE,
    class_weights=class_weights.tolist(),
    verbose=50,
    od_type="Iter",
    od_wait=30
)
cb_model.fit(X_train_imp, y_train, eval_set=(X_val_imp, y_val), use_best_model=True)
timing["CatBoost_train"] = time.time() - t0
print(f"CatBoost trained in {timing['CatBoost_train']:.1f}s\n")

# -----------------------
# 5d. Random Forest
# -----------------------
t0 = time.time()
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    n_jobs=-1
)
rf_model.fit(X_train_imp, y_train)
timing["RandomForest_train"] = time.time() - t0
print(f"Random Forest trained in {timing['RandomForest_train']:.1f}s\n")

# -----------------------
# 5e. FT-Transformer
# -----------------------
class SimpleFTTransformerNet(nn.Module):
    """Simple FT-Transformer-style model for continuous tabular features.

    Each numeric feature is projected into a token. A CLS token summarizes the row.
    This is intentionally simple and reproducible for reviewer-facing experiments.
    """
    def __init__(self, n_features, n_classes, d_token=64, n_heads=8, n_layers=3, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.feature_weight = nn.Parameter(torch.randn(n_features, d_token) * 0.02)
        self.feature_bias = nn.Parameter(torch.zeros(n_features, d_token))
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_token))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_token,
            nhead=n_heads,
            dim_feedforward=d_token * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_token),
            nn.Linear(d_token, n_classes)
        )

    def forward(self, x):
        # x: [batch, n_features]
        tokens = x.unsqueeze(-1) * self.feature_weight.unsqueeze(0) + self.feature_bias.unsqueeze(0)
        cls = self.cls_token.expand(x.size(0), -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        encoded = self.encoder(tokens)
        return self.head(encoded[:, 0])


class FTTransformerClassifier:
    """Sklearn-like wrapper that manages its own internal scaling to protect neural gradients."""
    def __init__(self, n_features, n_classes, random_state=42, epochs=20, batch_size=1024, lr=1e-3):
        self.n_features = n_features
        self.n_classes = n_classes
        self.random_state = random_state
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = lr
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.scaler = StandardScaler()  # Internal scaler specifically for deep learning stability
        self.model = SimpleFTTransformerNet(n_features, n_classes).to(self.device)

    def fit(self, X, y, X_val=None, y_val=None):
        torch.manual_seed(self.random_state)
        np.random.seed(self.random_state)

        # Scale locally right before tensor conversion to ensure neural network convergence
        X_np = self.scaler.fit_transform(X).astype("float32")
        y_np = np.asarray(y, dtype="int64")

        ds = TensorDataset(torch.tensor(X_np), torch.tensor(y_np))
        loader = DataLoader(ds, batch_size=self.batch_size, shuffle=True)

        classes = np.arange(self.n_classes)
        weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_np)
        weights = torch.tensor(weights, dtype=torch.float32, device=self.device)
        criterion = nn.CrossEntropyLoss(weight=weights)
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=1e-4)

        best_val_f1 = -1
        best_state = None
        patience = 5
        patience_left = patience

        for epoch in range(1, self.epochs + 1):
            self.model.train()
            total_loss = 0.0
            for xb, yb in loader:
                xb = xb.to(self.device)
                yb = yb.to(self.device)
                optimizer.zero_grad()
                loss = criterion(self.model(xb), yb)
                loss.backward()
                optimizer.step()
                total_loss += loss.item() * len(xb)

            msg = f"Epoch {epoch:02d}/{self.epochs} - loss={total_loss / len(ds):.4f}"
            if X_val is not None and y_val is not None:
                val_pred = self.predict(X_val)
                val_f1 = f1_score(y_val, val_pred, average="weighted", zero_division=0)
                msg += f" - val_weighted_f1={val_f1:.4f}"
                if val_f1 > best_val_f1:
                    best_val_f1 = val_f1
                    best_state = {k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()}
                    patience_left = patience
                else:
                    patience_left -= 1
                print(msg)
                if patience_left <= 0:
                    print("Early stopping FT-Transformer.")
                    break
            else:
                print(msg)

        if best_state is not None:
            self.model.load_state_dict(best_state)
        return self

    def predict_proba(self, X):
        # Apply the learned transformation mapping safely during evaluation paths
        X_np = self.scaler.transform(X).astype("float32")
        self.model.eval()
        out = []
        with torch.no_grad():
            for i in range(0, len(X_np), self.batch_size):
                xb = torch.tensor(X_np[i:i+self.batch_size], dtype=torch.float32, device=self.device)
                logits = self.model(xb)
                probs = torch.softmax(logits, dim=1).cpu().numpy()
                out.append(probs)
        return np.vstack(out)

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)


t0 = time.time()
ft_model = FTTransformerClassifier(
    n_features=X_train_imp.shape[1],
    n_classes=n_classes,
    random_state=RANDOM_STATE,
    epochs=20,
    batch_size=1024,
    lr=1e-3
)
# Train using the imputed inputs; wrapper internally scales them
ft_model.fit(X_train_imp, y_train, X_val=X_val_imp, y_val=y_val)
timing["FTTransformer_train"] = time.time() - t0
print(f"FT-Transformer trained in {timing['FTTransformer_train']:.1f}s on {ft_model.device}\n")

In [ ]:
from sklearn.metrics import f1_score, classification_report
import pandas as pd

def evaluate_model(name, model, X, y, label_encoder):
    # Features are already globally imputed (X_val_imp), so we predict directly.
    # Note: FT-Transformer will automatically handle its own scaling inside its wrapper.
    y_pred = model.predict(X)
    weighted_f1 = f1_score(y, y_pred, average="weighted")

    print(f"\n===== {name} =====")
    print(f"Weighted F1: {weighted_f1:.4f}")

    report = classification_report(
        y,
        y_pred,
        target_names=label_encoder.classes_,
        digits=4
    )
    print("\nPer-class F1:")
    print(report)

    return {
        "Model": name,
        "Weighted_F1": weighted_f1
    }

results = []

# All models evaluate uniformly using the native-scale, imputed validation dataset
results.append(evaluate_model("LightGBM", lgb_model, X_val_imp, y_val, le))
results.append(evaluate_model("XGBoost", xgb_model, X_val_imp, y_val, le))
results.append(evaluate_model("CatBoost", cb_model, X_val_imp, y_val, le))
results.append(evaluate_model("RandomForest", rf_model, X_val_imp, y_val, le))
results.append(evaluate_model("FT-Transformer", ft_model, X_val_imp, y_val, le))

results_df = pd.DataFrame(results)


print("\n=== Summary (Model Performance Across Imputed Validation Framework) ===")
print(results_df.sort_values("Weighted_F1", ascending=False))

## 6. Probability Calibration

In [ ]:
# Calibrate the primary model (LightGBM) using isotonic regression on the val set.
# This improves reliability of probability-based thresholds.

print("Calibrating LightGBM probabilities with isotonic regression...")
lgb_calibrated = CalibratedClassifierCV(
    lgb_model,
    method="isotonic",
    cv="prefit"  # use already-trained model
)
lgb_calibrated.fit(X_val, y_val)
print("Calibration complete.\n")

# Quick calibration check
raw_proba = lgb_model.predict_proba(X_val)
cal_proba = lgb_calibrated.predict_proba(X_val)

print("Raw LightGBM max confidence (mean):", raw_proba.max(axis=1).mean().round(4))
print("Calibrated LightGBM max confidence (mean):", cal_proba.max(axis=1).mean().round(4))

## 7. SHAP Diagnostics Only: No Feature Selection Used for Final Models

In [ ]:
# This cell computes initial SHAP rankings only for reporting and reviewer diagnostics.
X_shap_fs, y_shap_fs = None, None
initial_feature_importance_df = None

print("Computing initial SHAP ranking from the initial LightGBM model...")
explainer_initial = shap.TreeExplainer(lgb_model)
X_shap_fs, y_shap_fs = sample_df(X_train, y_train, MAX_SHAP_ROWS, RANDOM_STATE) if 'sample_df' in globals() else (X_train.sample(min(MAX_SHAP_ROWS, len(X_train)), random_state=RANDOM_STATE), None)
shap_raw_initial = explainer_initial.shap_values(X_shap_fs)

# Aggregate multi-class SHAP values into global feature importance.
def aggregate_multiclass_shap_importance(shap_values):
    if isinstance(shap_values, list):
        arr = np.stack(shap_values, axis=-1)  # [n, features, classes]
    else:
        arr = np.asarray(shap_values)
        if arr.ndim == 2:
            return np.mean(np.abs(arr), axis=0)
    return np.mean(np.abs(arr), axis=(0, 2))

importance = aggregate_multiclass_shap_importance(shap_raw_initial)
initial_feature_importance_df = pd.DataFrame({
    "feature": selected_features,
    "mean_abs_shap_initial": importance
}).sort_values("mean_abs_shap_initial", ascending=False).reset_index(drop=True)

print("Top initial SHAP features:")
display(initial_feature_importance_df.head(30))
initial_feature_importance_df.to_csv(os.path.join(DRIVE_RESULTS_PATH, "initial_shap_feature_ranking_no_ports.csv"), index=False)

plt.figure(figsize=(8, 6))
sns.barplot(
    data=initial_feature_importance_df.head(25),
    y="feature", x="mean_abs_shap_initial"
)
plt.title("SHAP Feature Ranking")
plt.tight_layout()
plt.show()

selected_features = numeric_feature_cols
print("Final selected_features remains full behavioral feature list:", len(selected_features))

## 8. Feature Selection Consistency Diagnostic Only

In [ ]:
# Compare mean vs median SHAP aggregation.
# It uses the initial trained full-feature LightGBM model only.

print("Running SHAP mean vs median aggregation diagnostic...")

# Use your existing SHAP sample
# Replace X_shap_fs if your variable name is different
X_shap_agg = X_shap_fs.copy()

# Existing trained LightGBM model
explainer = shap.TreeExplainer(lgb_model)

# Compute SHAP values
shap_raw = explainer.shap_values(X_shap_agg)

# Convert multiclass SHAP output
if isinstance(shap_raw, list):
    shap_arr = np.stack(shap_raw, axis=-1)
else:
    shap_arr = shap_raw

# Aggregate absolute SHAP across classes
# Shape: samples x features
shap_abs_per_sample_feature = np.mean(np.abs(shap_arr), axis=-1)

# Mean aggregation across samples
mean_importance = np.mean(shap_abs_per_sample_feature, axis=0)

# Median aggregation across samples
median_importance = np.median(shap_abs_per_sample_feature, axis=0)

feature_names = list(X_shap_agg.columns)

mean_median_shap_aggregation_df = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": mean_importance,
    "median_abs_shap": median_importance
})

# Ranking
mean_median_shap_aggregation_df["rank_mean"] = (
    mean_median_shap_aggregation_df["mean_abs_shap"]
    .rank(ascending=False, method="dense")
)

mean_median_shap_aggregation_df["rank_median"] = (
    mean_median_shap_aggregation_df["median_abs_shap"]
    .rank(ascending=False, method="dense")
)

# Spearman consistency
rank_corr = mean_median_shap_aggregation_df["mean_abs_shap"].corr(
    mean_median_shap_aggregation_df["median_abs_shap"],
    method="spearman"
)

print(f"Spearman consistency between mean and median SHAP aggregation: {rank_corr:.4f}")

display(
    mean_median_shap_aggregation_df
    .sort_values("rank_mean")
    .head(30)
    .reset_index(drop=True)
)

# Save results
mean_median_shap_aggregation_df.to_csv(
    os.path.join(
        DRIVE_RESULTS_PATH,
        "mean_vs_median_shap_aggregation_diagnostic.csv"
    ),
    index=False
)

print("Final pipeline")

## 9. Evaluation Utilities

In [ ]:
def fix_shape(y):
    y = np.asarray(y)

    # Convert (n, 1) -> (n,)
    if y.ndim == 2 and y.shape[1] == 1:
        y = y.ravel()

    return y


def evaluate_predictions(y_true, y_pred, label_encoder, title=""):
    y_true = fix_shape(y_true)
    y_pred = fix_shape(y_pred)

    print(f"\n===== {title} =====")
    print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
    print("Macro F1:", round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4))
    print("Weighted F1:", round(f1_score(y_true, y_pred, average="weighted", zero_division=0), 4))

    present_labels = sorted(np.unique(np.concatenate([y_true, y_pred])))
    present_target_names = [label_encoder.classes_[i] for i in present_labels]

    print("\nClassification Report:")
    print(classification_report(
        y_true, y_pred,
        labels=present_labels,
        target_names=present_target_names,
        zero_division=0
    ))


def collect_metrics(y_true, y_pred, model_name, dataset_name):
    y_true = fix_shape(y_true)
    y_pred = fix_shape(y_pred)

    present_labels = np.unique(np.concatenate([y_true, y_pred]))

    return {
        "Model": model_name,
        "Dataset": dataset_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro_Precision": precision_score(y_true, y_pred, labels=present_labels, average="macro", zero_division=0),
        "Macro_Recall": recall_score(y_true, y_pred, labels=present_labels, average="macro", zero_division=0),
        "Macro_F1": f1_score(y_true, y_pred, labels=present_labels, average="macro", zero_division=0),
        "Weighted_Precision": precision_score(y_true, y_pred, labels=present_labels, average="weighted", zero_division=0),
        "Weighted_Recall": recall_score(y_true, y_pred, labels=present_labels, average="weighted", zero_division=0),
        "Weighted_F1": f1_score(y_true, y_pred, labels=present_labels, average="weighted", zero_division=0),
    }

## 10. Model Evaluation (All Models x 4 Datasets)

In [ ]:
import time
import pandas as pd

# Define model dict for iteration - all models run on the native, imputed scale!
models = {
    "LightGBM": lgb_model,
    "XGBoost": xgb_model,
    "CatBoost": cb_model,
    "RandomForest": rf_model,
    "FT-Transformer": ft_model,
}

# The raw datasets you want to evaluate
raw_datasets = {
    "Validation": (X_val, y_val),
    "Host-Test": (X_host_test, y_host_test),
    "Time-Test": (X_time_test, y_time_test),
    "Hard-Test": (X_hard_test, y_hard_test),
}

metrics = []
inference_times = []

# Loop through models and datasets uniformly
for model_name, model in models.items():
    for ds_name, (X_ds_raw, y_ds) in raw_datasets.items():

        # 1. Handle missing values uniformly across all test sets.
        # Leaving the data on its native scale completely protects your F1 metrics.
        X_ds_imp = pd.DataFrame(
            global_imputer.transform(X_ds_raw),
            columns=X_ds_raw.columns
        )

        # 2. Track Inference Speeds on the clean, native-scale framework
        # Note: FT-Transformer seamlessly scales this internally without extra code here
        t0 = time.time()
        y_pred = fix_shape(model.predict(X_ds_imp))
        t_infer = time.time() - t0

        # 3. Log Performance Metrics
        metrics.append(collect_metrics(y_ds, y_pred, model_name, ds_name))
        inference_times.append({
            "Model": model_name,
            "Dataset": ds_name,
            "Samples": len(X_ds_imp),
            "Time_s": round(t_infer, 4),
            "Throughput_sps": round(len(X_ds_imp) / max(t_infer, 1e-6), 0)
        })

        evaluate_predictions(y_ds, y_pred, le, f"{model_name} - {ds_name}")

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

def plot_confusion(y_true, y_pred, label_encoder, title):
    cm = confusion_matrix(y_true, y_pred)
    labels = label_encoder.classes_

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=labels,
        yticklabels=labels
    )
    plt.title(title)
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()


# Macro F1 comparison across all models and datasets
metrics_df = pd.DataFrame(metrics)

plt.figure(figsize=(12, 6))
sns.barplot(data=metrics_df, x="Dataset", y="Macro_F1", hue="Model")
plt.title("Attack Detection Performance (Macro F1) - All Models")
plt.ylabel("Macro F1 Score")
plt.xlabel("Dataset")
plt.legend(title="Model")
plt.tight_layout()
plt.show()


# Confusion matrices for best model on test sets (Updated to match raw_datasets names and handle imputation)
for ds_name in ["Time-Test", "Hard-Test"]:
    X_ds_raw, y_ds = raw_datasets[ds_name]

    # Clean the evaluation matrix uniformly using the global imputer on its native scale
    X_ds_imp = pd.DataFrame(
        global_imputer.transform(X_ds_raw),
        columns=X_ds_raw.columns
    )

    y_pred = lgb_model.predict(X_ds_imp)
    plot_confusion(y_ds, y_pred, le, f"LightGBM Confusion Matrix - {ds_name}")

## 11. SHAP Signatures & Explanation Stability

In [ ]:
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt

def sample_df(X, y, max_rows, random_state=42):
    if len(X) <= max_rows:
        return X.copy(), np.asarray(y)
    rng = np.random.default_rng(random_state)
    idx = rng.choice(len(X), size=max_rows, replace=False)
    return X.iloc[idx].copy(), np.asarray(y)[idx]


def convert_shap_to_predicted_class_vectors(shap_values, pred_class_idx):
    """Extract SHAP vector for the predicted class of each sample."""
    if isinstance(shap_values, list):
        return np.array([shap_values[c][i] for i, c in enumerate(pred_class_idx)])

    shap_values = np.array(shap_values)
    if shap_values.ndim == 3:
        return np.array([shap_values[i, :, c] for i, c in enumerate(pred_class_idx)])
    if shap_values.ndim == 2:
        return shap_values
    raise ValueError(f"Unexpected SHAP shape: {shap_values.shape}")


def l2_normalize_rows(A, eps=1e-12):
    """Normalizes rows to a unit hypersphere to completely eliminate input range dominance."""
    A = np.asarray(A, dtype=float)
    denom = np.linalg.norm(A, axis=1, keepdims=True)
    denom = np.maximum(denom, eps)
    return A / denom


def compute_shap_vectors(X, explainer, model, normalize_rows=True):
    """Compute SHAP vectors for predicted class, with optional row-wise L2 normalization."""
    shap_raw = explainer.shap_values(X)
    pred_idx = np.argmax(model.predict_proba(X), axis=1)
    vecs = convert_shap_to_predicted_class_vectors(shap_raw, pred_idx)
    raw_vecs = vecs.copy()
    if normalize_rows:
        vecs = l2_normalize_rows(vecs)
    return vecs, pred_idx, raw_vecs


# Create SHAP explainer on the initial LightGBM model operating on raw native feature values
explainer = shap.TreeExplainer(lgb_model)

# Run an inline dataset preprocessing alignment to handle training imputation safely
X_train_imp = pd.DataFrame(global_imputer.transform(X_train), columns=X_train.columns)

shap_data = {}
# Updated loop to correctly call "raw_datasets" and apply uniform imputation transformations
for name, (X_ds, y_ds) in [("Train", (X_train_imp, y_train))] + [
    (k, (pd.DataFrame(global_imputer.transform(v[0]), columns=v[0].columns), v[1]))
    for k, v in raw_datasets.items()
]:
    X_s, y_s = sample_df(X_ds, y_ds, MAX_SHAP_ROWS, RANDOM_STATE)
    vecs_norm, pred_idx, vecs_raw = compute_shap_vectors(X_s, explainer, lgb_model, normalize_rows=True)
    shap_data[name] = {
        "X": X_s,
        "y": y_s,
        "vecs": vecs_norm,          # normalized SHAP vectors for similarity metrics
        "vecs_raw": vecs_raw,       # raw SHAP vectors for sensitivity/diagnostics
        "pred_idx": pred_idx
    }
    print(f"{name}: normalized SHAP shape = {vecs_norm.shape}; raw SHAP shape = {vecs_raw.shape}")

# Summary plot uses raw SHAP values for readability (unaltered explanation space)
plt.figure()
train_shap_raw = explainer.shap_values(shap_data["Train"]["X"])
shap.summary_plot(train_shap_raw, shap_data["Train"]["X"], show=False, max_display=25)
plt.tight_layout()
plt.show()

In [ ]:
def build_class_signatures(shap_vectors, y_true, label_encoder, min_rows=10):
    """Build normalized mean SHAP signature per class."""
    signatures = {}
    counts = {}
    for class_idx in np.unique(y_true):
        mask = (y_true == class_idx)
        if mask.sum() >= min_rows:
            class_name = label_encoder.inverse_transform([class_idx])[0]
            # Mean of normalized row-level SHAP vectors.
            sig = np.mean(shap_vectors[mask], axis=0)
            sig = sig / max(np.linalg.norm(sig), 1e-12)
            signatures[class_name] = sig
            counts[class_name] = int(mask.sum())
    return signatures, counts


def compute_signature_variance(shap_vectors, y_true, label_encoder, signatures, min_rows=10):
    """Account for signature heterogeneity using intra-class variance and cosine dispersion."""
    rows = []
    for class_idx in np.unique(y_true):
        mask = (y_true == class_idx)
        if mask.sum() < min_rows:
            continue
        class_name = label_encoder.inverse_transform([class_idx])[0]
        if class_name not in signatures:
            continue
        class_vecs = shap_vectors[mask]
        sig = signatures[class_name].reshape(1, -1)
        sims = cosine_similarity(class_vecs, sig).ravel()
        rows.append({
            "Class": class_name,
            "n": int(mask.sum()),
            "mean_intra_class_cosine": float(np.mean(sims)),
            "std_intra_class_cosine": float(np.std(sims)),
            "min_intra_class_cosine": float(np.min(sims)),
            "mean_feature_variance": float(np.mean(np.var(class_vecs, axis=0))),
            "median_feature_variance": float(np.median(np.var(class_vecs, axis=0))),
            "heterogeneity_flag": bool(np.mean(sims) < 0.70 or np.std(sims) > 0.25)
        })
    return pd.DataFrame(rows).sort_values(["heterogeneity_flag", "mean_intra_class_cosine"], ascending=[False, True])


def top_k_signature_features(signature_vector, feature_names, k=8):
    idx = np.argsort(np.abs(signature_vector))[-k:][::-1]
    return [(feature_names[i], float(signature_vector[i])) for i in idx]


# Build signatures and variance tables for each split
signatures = {}
signature_counts = {}
signature_variance_tables = {}

for name in shap_data:
    sigs, counts = build_class_signatures(
        shap_data[name]["vecs"], shap_data[name]["y"],
        le, MIN_ROWS_PER_CLASS_FOR_SIGNATURE
    )
    signatures[name] = sigs
    signature_counts[name] = counts
    variance_df = compute_signature_variance(
        shap_data[name]["vecs"], shap_data[name]["y"], le, sigs,
        MIN_ROWS_PER_CLASS_FOR_SIGNATURE
    )
    signature_variance_tables[name] = variance_df
    variance_df.to_csv(os.path.join(DRIVE_RESULTS_PATH, f"{name.lower()}_signature_variance_heterogeneity.csv"), index=False)

    print(f"\n{name} signatures: {list(sigs.keys())}")
    print(f"{name} signature heterogeneity / variance analysis:")
    display(variance_df.round(4))

# Print top features for training signatures
print("\nTop features per training signature:")
for attack_name, sig in list(signatures["Train"].items())[:20]:
    print(f"\n  {attack_name}:")
    for feat, val in top_k_signature_features(sig, selected_features, k=TOP_K_SIGNATURE_FEATURES):
        print(f"    {feat:35s} {val:+.5f}")

In [ ]:
def signature_similarity_table(train_sigs, test_sigs):
    rows = []
    for name, train_sig in train_sigs.items():
        if name not in test_sigs:
            continue
        same = cosine_similarity([train_sig], [test_sigs[name]])[0, 0]
        other_sims = [
            cosine_similarity([train_sig], [other_sig])[0, 0]
            for other_name, other_sig in test_sigs.items()
            if other_name != name
        ]
        mean_other = np.mean(other_sims) if other_sims else np.nan
        rows.append({
            "Attack": name,
            "SameClassSimilarity": same,
            "OtherClassMeanSimilarity": mean_other,
            "Gap": same - mean_other if not np.isnan(mean_other) else np.nan,
            "Interpretation": "stable/distinct" if (not np.isnan(mean_other) and same - mean_other >= 0.25) else "weak/overlapping"
        })
    if not rows:
        return pd.DataFrame(columns=["Attack", "SameClassSimilarity", "OtherClassMeanSimilarity", "Gap", "Interpretation"])
    return pd.DataFrame(rows).sort_values("Gap", ascending=False)


all_similarity_tables = {}
for test_name in ["Host-Test", "Time-Test", "Hard-Test"]:
    sim_table = signature_similarity_table(signatures["Train"], signatures.get(test_name, {}))
    all_similarity_tables[test_name] = sim_table
    print(f"\nTrain vs {test_name}:")
    display(sim_table.round(4))
    sim_table.to_csv(os.path.join(DRIVE_RESULTS_PATH, f"{test_name.lower()}_signature_stability_normalized.csv"), index=False)


# Normalization sensitivity: compare raw-signature cosine vs normalized-signature cosine.
def build_raw_signatures(raw_vectors, y_true, label_encoder, min_rows=10):
    sigs = {}
    for class_idx in np.unique(y_true):
        mask = (y_true == class_idx)
        if mask.sum() >= min_rows:
            class_name = label_encoder.inverse_transform([class_idx])[0]
            sig = np.mean(raw_vectors[mask], axis=0)
            sigs[class_name] = sig
    return sigs

raw_signatures = {
    name: build_raw_signatures(shap_data[name]["vecs_raw"], shap_data[name]["y"], le, MIN_ROWS_PER_CLASS_FOR_SIGNATURE)
    for name in shap_data
}

normalization_sensitivity_rows = []
for test_name in ["Host-Test", "Time-Test", "Hard-Test"]:
    raw_table = signature_similarity_table(raw_signatures["Train"], raw_signatures.get(test_name, {}))
    norm_table = all_similarity_tables[test_name]
    merged = raw_table[["Attack", "SameClassSimilarity", "Gap"]].merge(
        norm_table[["Attack", "SameClassSimilarity", "Gap"]],
        on="Attack", suffixes=("_raw", "_normalized")
    )
    merged["Dataset"] = test_name
    normalization_sensitivity_rows.append(merged)

normalization_sensitivity_df = pd.concat(normalization_sensitivity_rows, ignore_index=True) if normalization_sensitivity_rows else pd.DataFrame()
print("\nSHAP normalization sensitivity table:")
display(normalization_sensitivity_df.round(4))
normalization_sensitivity_df.to_csv(os.path.join(DRIVE_RESULTS_PATH, "shap_normalization_sensitivity.csv"), index=False)

# Signature similarity matrix (heatmap)
train_sigs = signatures["Train"]
attack_names = list(train_sigs.keys())
if len(attack_names) > 1:
    matrix = np.array([
        [cosine_similarity([train_sigs[a]], [train_sigs[b]])[0, 0]
         for b in attack_names]
        for a in attack_names
    ])
    plt.figure(figsize=(10, 8))
    sns.heatmap(pd.DataFrame(matrix, index=attack_names, columns=attack_names),
                cmap="coolwarm", center=0.5)
    plt.title("Normalized Attack Signature Similarity Matrix (Train)")
    plt.tight_layout()
    plt.show()

## 12. Safe Decision Policy (Enhanced)

In [ ]:
assert benign_idx is not None, "A benign class is required for the safe-decision layer."


# Store train signatures by class index
train_sig_by_class_idx = {}
for label_name, sig in signatures["Train"].items():
    class_idx = le.transform([label_name])[0]
    train_sig_by_class_idx[class_idx] = sig


def compute_pred_similarities_vectorized(shap_vecs, pred_idx, train_sig_by_class_idx):
    """Vectorized cosine similarity between each sample's SHAP vector
    and the training signature of its predicted class.

    ~10-50x faster than row-by-row computation.
    """
    n = len(shap_vecs)
    sims = np.full(n, -1.0)

    for class_idx, sig in train_sig_by_class_idx.items():
        mask = (pred_idx == class_idx)
        if mask.any():
            class_vecs = shap_vecs[mask]
            # Vectorized cosine similarity: (n_samples, n_features) vs (1, n_features)
            sims[mask] = cosine_similarity(class_vecs, sig.reshape(1, -1)).ravel()

    return sims


def evaluate_safe_decision_policy(proba, pred_idx, true_idx, benign_idx,
                                  pred_signature_similarity,
                                  prob_threshold, sim_threshold):
    """Three-tier safe decision: ALLOW / BLOCK / ESCALATE."""
    attack_conf = 1 - proba[:, benign_idx]

    decisions = np.full(len(true_idx), "ESCALATE", dtype=object)

    # ALLOW: predicted benign with low attack confidence
    allow_mask = (pred_idx == benign_idx) & (attack_conf < prob_threshold)
    decisions[allow_mask] = "ALLOW"

    # BLOCK: predicted attack with high confidence AND high signature similarity
    block_mask = (
        (pred_idx != benign_idx) &
        (attack_conf >= prob_threshold) &
        (pred_signature_similarity >= sim_threshold)
    )
    decisions[block_mask] = "BLOCK"

    # Everything else remains ESCALATE

    true_attack = (true_idx != benign_idx)
    true_benign = ~true_attack

    n_true_attack = max(np.sum(true_attack), 1)
    n_true_benign = max(np.sum(true_benign), 1)

    missed_attacks = int(np.sum((decisions == "ALLOW") & true_attack))
    attack_not_missed_rate = np.sum((decisions != "ALLOW") & true_attack) / n_true_attack
    false_block_rate = np.sum((decisions == "BLOCK") & true_benign) / n_true_benign
    escalation_rate = np.mean(decisions == "ESCALATE")

    # Per-class metrics
    per_class = {}
    for c in np.unique(true_idx):
        c_mask = (true_idx == c)
        c_name = le.inverse_transform([c])[0]
        if c == benign_idx:
            per_class[c_name] = {
                "n": int(c_mask.sum()),
                "correctly_allowed": int(np.sum((decisions == "ALLOW") & c_mask)),
                "false_blocked": int(np.sum((decisions == "BLOCK") & c_mask)),
                "escalated": int(np.sum((decisions == "ESCALATE") & c_mask)),
            }
        else:
            per_class[c_name] = {
                "n": int(c_mask.sum()),
                "missed": int(np.sum((decisions == "ALLOW") & c_mask)),
                "correctly_blocked": int(np.sum((decisions == "BLOCK") & c_mask)),
                "escalated": int(np.sum((decisions == "ESCALATE") & c_mask)),
            }

    return {
        "missed_attacks": missed_attacks,
        "attack_not_missed_rate": attack_not_missed_rate,
        "false_block_rate": false_block_rate,
        "escalation_rate": escalation_rate,
        "per_class": per_class,
    }


print("Decision functions defined (vectorized).")

## 13. Automatic Threshold Search (Constrained)

In [ ]:
# Prepare validation SHAP + confidence for threshold search

X_val_shap, y_val_shap = sample_df(X_val, y_val, MAX_SHAP_ROWS, RANDOM_STATE)
val_proba = lgb_calibrated.predict_proba(X_val_shap)  # Use CALIBRATED probabilities
val_pred_idx = np.argmax(val_proba, axis=1)
val_attack_conf = 1 - val_proba[:, benign_idx]

val_shap_raw = explainer.shap_values(X_val_shap)
val_shap_vecs = convert_shap_to_predicted_class_vectors(val_shap_raw, val_pred_idx)

# Vectorized similarity
val_sim = compute_pred_similarities_vectorized(val_shap_vecs, val_pred_idx, train_sig_by_class_idx)

correct_attack_mask = (val_pred_idx == y_val_shap) & (y_val_shap != benign_idx)


# Grid search with MINIMUM DETECTION RATE CONSTRAINT

prob_quantiles = [0.30, 0.40, 0.50, 0.60, 0.70, 0.80]
sim_quantiles = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]

search_rows = []

for pq in prob_quantiles:
    prob_threshold_candidate = np.quantile(val_attack_conf, pq)

    for sq in sim_quantiles:
        if np.sum(correct_attack_mask) > 10:
            sim_threshold_candidate = np.quantile(val_sim[correct_attack_mask], sq)
        else:
            sim_threshold_candidate = np.quantile(val_sim, sq)

        result = evaluate_safe_decision_policy(
            proba=val_proba,
            pred_idx=val_pred_idx,
            true_idx=y_val_shap,
            benign_idx=benign_idx,
            pred_signature_similarity=val_sim,
            prob_threshold=prob_threshold_candidate,
            sim_threshold=sim_threshold_candidate
        )

        search_rows.append({
            "prob_quantile": pq,
            "sim_quantile": sq,
            "prob_threshold": prob_threshold_candidate,
            "sim_threshold": sim_threshold_candidate,
            "missed_attacks": result["missed_attacks"],
            "attack_not_missed_rate": result["attack_not_missed_rate"],
            "false_block_rate": result["false_block_rate"],
            "escalation_rate": result["escalation_rate"],
        })

threshold_search_df = pd.DataFrame(search_rows)

# *** CRITICAL FIX: Enforce minimum attack detection rate ***
valid_df = threshold_search_df[
    threshold_search_df["attack_not_missed_rate"] >= MIN_ATTACK_DETECTION_RATE
].copy()

if len(valid_df) == 0:
    print(f"WARNING: No threshold pair meets MIN_ATTACK_DETECTION_RATE={MIN_ATTACK_DETECTION_RATE}")
    print("Relaxing to best available attack_not_missed_rate...")
    valid_df = threshold_search_df.copy()

# Sort: lowest false_block_rate, then lowest escalation_rate
valid_df = valid_df.sort_values(
    by=["false_block_rate", "escalation_rate", "missed_attacks"],
    ascending=[True, True, True]
).reset_index(drop=True)

best_row = valid_df.iloc[0]
prob_threshold = best_row["prob_threshold"]
sim_threshold = best_row["sim_threshold"]

print("Best thresholds (with detection rate >= {:.0%}):".format(MIN_ATTACK_DETECTION_RATE))
print(f"  prob_threshold = {prob_threshold:.6f}  (quantile {best_row['prob_quantile']})")
print(f"  sim_threshold  = {sim_threshold:.6f}  (quantile {best_row['sim_quantile']})")
print(f"  attack_not_missed_rate = {best_row['attack_not_missed_rate']:.4f}")
print(f"  false_block_rate       = {best_row['false_block_rate']:.4f}")
print(f"  escalation_rate        = {best_row['escalation_rate']:.4f}")

print("\nFull search results (filtered):")
display(valid_df.round(4))

## 14. Test Safe Decision Policy

In [ ]:
def test_safe_policy(X_test, y_test, name, prob_threshold, sim_threshold):
    """Evaluate safe decision policy on a test set with timing."""
    X_test_small, y_test_small = sample_df(X_test, y_test, MAX_SHAP_ROWS, RANDOM_STATE)

    # Prediction timing
    t0 = time.time()
    proba = lgb_calibrated.predict_proba(X_test_small)
    t_pred = time.time() - t0

    pred_idx = np.argmax(proba, axis=1)

    # SHAP timing
    t0 = time.time()
    shap_raw = explainer.shap_values(X_test_small)
    shap_vecs = convert_shap_to_predicted_class_vectors(shap_raw, pred_idx)
    t_shap = time.time() - t0

    # Similarity timing
    t0 = time.time()
    pred_sim = compute_pred_similarities_vectorized(shap_vecs, pred_idx, train_sig_by_class_idx)
    t_sim = time.time() - t0

    # Baseline (confidence only, no SHAP)
    attack_conf = 1 - proba[:, benign_idx]
    baseline_pred_attack = attack_conf >= prob_threshold
    true_attack = y_test_small != benign_idx
    true_benign = ~true_attack

    baseline_missed = int(np.sum((~baseline_pred_attack) & true_attack))
    baseline_not_missed_rate = np.sum(baseline_pred_attack & true_attack) / max(np.sum(true_attack), 1)
    baseline_false_block_rate = np.sum(baseline_pred_attack & true_benign) / max(np.sum(true_benign), 1)

    # Explain-aware
    safe = evaluate_safe_decision_policy(
        proba=proba, pred_idx=pred_idx, true_idx=y_test_small,
        benign_idx=benign_idx, pred_signature_similarity=pred_sim,
        prob_threshold=prob_threshold, sim_threshold=sim_threshold
    )

    # Print per-class breakdown
    print(f"\n--- {name}: Per-Class Safe Decision Breakdown ---")
    for cls_name, cls_info in safe["per_class"].items():
        print(f"  {cls_name}: {cls_info}")

    total_time = t_pred + t_shap + t_sim
    n = len(X_test_small)

    return pd.DataFrame([{
        "Dataset": name,
        "prob_threshold": prob_threshold,
        "sim_threshold": sim_threshold,
        "Baseline_missed_attacks": baseline_missed,
        "Baseline_attack_not_missed_rate": baseline_not_missed_rate,
        "Baseline_false_block_rate": baseline_false_block_rate,
        "ExplainAware_missed_attacks": safe["missed_attacks"],
        "ExplainAware_attack_not_missed_rate": safe["attack_not_missed_rate"],
        "ExplainAware_false_block_rate": safe["false_block_rate"],
        "ExplainAware_escalation_rate": safe["escalation_rate"],
        "Time_prediction_s": round(t_pred, 3),
        "Time_SHAP_s": round(t_shap, 3),
        "Time_similarity_s": round(t_sim, 3),
        "Time_total_s": round(total_time, 3),
        "Throughput_decisions_per_s": round(n / max(total_time, 1e-6), 0),
    }])


# Apply to all test sets
safe_results = pd.concat([
    test_safe_policy(X_host_test, y_host_test, "Host-Test", prob_threshold, sim_threshold),
    test_safe_policy(X_time_test, y_time_test, "Time-Test", prob_threshold, sim_threshold),
    test_safe_policy(X_hard_test, y_hard_test, "Hard-Test", prob_threshold, sim_threshold),
], ignore_index=True)

print("\nSafe Decision Results:")
display(safe_results.round(4))

## 15. LIME Cross-Validation of SHAP Explanations

In [ ]:
if LIME_AVAILABLE:
    print("Running LIME cross-validation on a sample of predictions...\n")

    # Create LIME explainer on training data
    X_train_np = X_train.fillna(X_train.median()).values
    lime_explainer = lime.lime_tabular.LimeTabularExplainer(
        X_train_np,
        feature_names=selected_features,
        class_names=[str(c) for c in le.classes_],
        mode="classification",
        random_state=RANDOM_STATE,
        discretize_continuous=True
    )

    # Sample a small set for LIME (expensive per-instance)
    N_LIME = 50
    X_lime, y_lime = sample_df(X_host_test, y_host_test, N_LIME, RANDOM_STATE)
    lime_pred_idx = np.argmax(lgb_model.predict_proba(X_lime), axis=1)

    # Get SHAP for the same samples
    lime_shap_raw = explainer.shap_values(X_lime)
    lime_shap_vecs = convert_shap_to_predicted_class_vectors(lime_shap_raw, lime_pred_idx)

    agreement_scores = []
    for i in range(len(X_lime)):
        # SHAP top features
        shap_abs = np.abs(lime_shap_vecs[i])
        shap_top5 = set(np.argsort(shap_abs)[-5:])

        # LIME top features
        row = X_lime.iloc[i].fillna(0).values
        lime_exp = lime_explainer.explain_instance(
            row, lgb_model.predict_proba,
            num_features=5, labels=[int(lime_pred_idx[i])]
        )
        lime_feats = lime_exp.as_map().get(int(lime_pred_idx[i]), [])
        lime_top5 = set([f[0] for f in lime_feats[:5]])

        # Jaccard overlap
        if len(shap_top5 | lime_top5) > 0:
            jaccard = len(shap_top5 & lime_top5) / len(shap_top5 | lime_top5)
        else:
            jaccard = 0.0
        agreement_scores.append(jaccard)

    agreement_scores = np.array(agreement_scores)
    print(f"SHAP-LIME Agreement (Jaccard of top-5 features):")
    print(f"  Mean:   {agreement_scores.mean():.3f}")
    print(f"  Median: {np.median(agreement_scores):.3f}")
    print(f"  Std:    {agreement_scores.std():.3f}")
    print(f"  Min:    {agreement_scores.min():.3f}")
    print(f"  Max:    {agreement_scores.max():.3f}")

    # Low agreement samples -> candidates for ESCALATE
    low_agreement = agreement_scores < 0.2
    print(f"\n  Samples with low SHAP-LIME agreement (<0.2): {low_agreement.sum()} / {len(agreement_scores)}")
    print("  These samples should be flagged for ESCALATION in production.")

    plt.figure(figsize=(8, 4))
    plt.hist(agreement_scores, bins=20, edgecolor="black", alpha=0.7)
    plt.xlabel("SHAP-LIME Top-5 Jaccard Agreement")
    plt.ylabel("Count")
    plt.title("Distribution of SHAP-LIME Explanation Agreement")
    plt.tight_layout()
    plt.show()
else:
    print("LIME not available. Skipping cross-validation.")
    print("Install with: pip install lime")

## 16. Visualization

In [ ]:
# Decision policy comparison plot
safe_plot = safe_results.melt(
    id_vars="Dataset",
    value_vars=[
        "Baseline_attack_not_missed_rate",
        "ExplainAware_attack_not_missed_rate"
    ],
    var_name="Method",
    value_name="AttackDetectionRate"
)
safe_plot["Method"] = safe_plot["Method"].map({
    "Baseline_attack_not_missed_rate": "Baseline (confidence only)",
    "ExplainAware_attack_not_missed_rate": "Explain-Aware (SHAP signatures)"
})

plt.figure(figsize=(10, 6))
sns.barplot(data=safe_plot, x="Dataset", y="AttackDetectionRate", hue="Method")
plt.title("Decision Policy Comparison: Baseline vs Explain-Aware")
plt.ylabel("Attack Detection Rate")
plt.tight_layout()
plt.show()

# Missed attack reduction
comparison = safe_results.copy()
comparison["Missed_Attack_Reduction"] = (
    comparison["Baseline_missed_attacks"] - comparison["ExplainAware_missed_attacks"]
)

plt.figure(figsize=(8, 5))
sns.barplot(data=comparison, x="Dataset", y="Missed_Attack_Reduction")
plt.title("Reduction in Missed Attacks Using Explanations")
plt.tight_layout()
plt.show()

# False block rate comparison
fbr_plot = safe_results.melt(
    id_vars="Dataset",
    value_vars=["Baseline_false_block_rate", "ExplainAware_false_block_rate"],
    var_name="Method", value_name="FalseBlockRate"
)
fbr_plot["Method"] = fbr_plot["Method"].map({
    "Baseline_false_block_rate": "Baseline",
    "ExplainAware_false_block_rate": "Explain-Aware"
})

plt.figure(figsize=(10, 6))
sns.barplot(data=fbr_plot, x="Dataset", y="FalseBlockRate", hue="Method")
plt.title("False Block Rate Comparison")
plt.tight_layout()
plt.show()

## 17. Multi-Seed Robustness Evaluation

In [ ]:
# Run decision policy evaluation with different random seeds
# to compute confidence intervals

print(f"Running multi-seed evaluation with seeds: {RANDOM_SEEDS}\n")

multiseed_results = []

for seed in RANDOM_SEEDS:
    for ds_name, (X_ds, y_ds) in [
        ("Host-Test", (X_host_test, y_host_test)),
        ("Time-Test", (X_time_test, y_time_test)),
        ("Hard-Test", (X_hard_test, y_hard_test)),
    ]:
        X_s, y_s = sample_df(X_ds, y_ds, MAX_SHAP_ROWS, seed)

        proba = lgb_calibrated.predict_proba(X_s)
        pred_idx = np.argmax(proba, axis=1)

        shap_raw = explainer.shap_values(X_s)
        shap_vecs = convert_shap_to_predicted_class_vectors(shap_raw, pred_idx)
        pred_sim = compute_pred_similarities_vectorized(shap_vecs, pred_idx, train_sig_by_class_idx)

        safe = evaluate_safe_decision_policy(
            proba=proba, pred_idx=pred_idx, true_idx=y_s,
            benign_idx=benign_idx, pred_signature_similarity=pred_sim,
            prob_threshold=prob_threshold, sim_threshold=sim_threshold
        )

        multiseed_results.append({
            "Seed": seed,
            "Dataset": ds_name,
            "attack_not_missed_rate": safe["attack_not_missed_rate"],
            "false_block_rate": safe["false_block_rate"],
            "escalation_rate": safe["escalation_rate"],
            "missed_attacks": safe["missed_attacks"],
        })

multiseed_df = pd.DataFrame(multiseed_results)

# Summary with mean +/- std
summary = multiseed_df.groupby("Dataset").agg(
    attack_detection_mean=("attack_not_missed_rate", "mean"),
    attack_detection_std=("attack_not_missed_rate", "std"),
    false_block_mean=("false_block_rate", "mean"),
    false_block_std=("false_block_rate", "std"),
    escalation_mean=("escalation_rate", "mean"),
    escalation_std=("escalation_rate", "std"),
).round(4)

print("Multi-seed results (mean +/- std):")
display(summary)
display(multiseed_df.round(4))

## 18. Save All Results & Deployment Artifact

In [ ]:


timing_df = pd.DataFrame(inference_times)

# Save tables
metrics_df.to_csv(os.path.join(DRIVE_RESULTS_PATH, "model_detection_metrics.csv"), index=False)
safe_results.to_csv(os.path.join(DRIVE_RESULTS_PATH, "safe_decision_results.csv"), index=False)
timing_df.to_csv(os.path.join(DRIVE_RESULTS_PATH, "inference_timing.csv"), index=False)
multiseed_df.to_csv(os.path.join(DRIVE_RESULTS_PATH, "multiseed_results.csv"), index=False)
threshold_search_df.to_csv(os.path.join(DRIVE_RESULTS_PATH, "threshold_search.csv"), index=False)

# Save models
joblib.dump(lgb_model, os.path.join(DRIVE_RESULTS_PATH, "lightgbm_model_initial_no_ports.pkl"))
joblib.dump(xgb_model, os.path.join(DRIVE_RESULTS_PATH, "xgboost_model_initial_no_ports.pkl"))
joblib.dump(cb_model, os.path.join(DRIVE_RESULTS_PATH, "catboost_model_initial_no_ports.pkl"))
joblib.dump(rf_model, os.path.join(DRIVE_RESULTS_PATH, "randomforest_model_initial_no_ports.pkl"))
joblib.dump(ft_model, os.path.join(DRIVE_RESULTS_PATH, "ft_transformer_model_initial_no_ports.pkl"))

# Unified deployment artifact uses initial LightGBM only for explanation-guided decision policy.
# Updated "rf_imputer" to "global_imputer" to reflect the pipeline alignment
deployment_artifact = {
    "lgb_model": lgb_model,
    "lgb_calibrated": lgb_calibrated,
    "explainer_model": lgb_model,
    "label_encoder": le,
    "selected_features": selected_features,
    "train_sig_by_class_idx": train_sig_by_class_idx,
    "prob_threshold": float(prob_threshold),
    "sim_threshold": float(sim_threshold),
    "benign_idx": int(benign_idx),
    "global_imputer": global_imputer,
    "dropped_columns": [c for c in leakage_cols if c in train_df.columns],
    "shap_signature_normalization": "row-wise L2 normalization, then class mean, then signature L2 normalization",
}
joblib.dump(deployment_artifact, os.path.join(DRIVE_RESULTS_PATH, "deployment_artifact_initial_no_ports.pkl"))

# Save config
config = {
    "dataset": "BCCC-CIC-IDS-2017",
    "models": ["LightGBM", "XGBoost", "CatBoost", "RandomForest", "FT-Transformer"],
    "random_seeds": RANDOM_SEEDS,
    "feature_selection_used_for_final_models": False,
    "feature_selection_consistency_diagnostic": RUN_FEATURE_SELECTION_CONSISTENCY_DIAGNOSTIC,
    "selected_features": selected_features,
    "dropped_columns": [c for c in leakage_cols if c in train_df.columns],
    "benign_files_kept": sorted(list(KEEP_BENIGN_FILES)),
    "top_k_signature_features": TOP_K_SIGNATURE_FEATURES,
    "min_rows_per_class_for_signature": MIN_ROWS_PER_CLASS_FOR_SIGNATURE,
    "max_shap_rows": MAX_SHAP_ROWS,
    "min_class_count": MIN_CLASS_COUNT,
    "seen_host_ratio": SEEN_HOST_RATIO,
    "prob_threshold": float(prob_threshold),
    "sim_threshold": float(sim_threshold),
    "min_attack_detection_rate": MIN_ATTACK_DETECTION_RATE,
    "calibration": "isotonic",
    "xai_methods": ["SHAP TreeExplainer", "LIME cross-validation if available"],
    "reviewer_fixes": [
        "Dropped src_port and dst_port",
        "No feature selection/retraining for final results",
        "Replaced MLP with FT-Transformer",
        "Kept only friday_benign.csv and thursday_benign.csv benign files",
        "Added SHAP normalization sensitivity via post-hoc L2 Vector Normalization",
        "Added signature variance and heterogeneity analysis",
        "Added optional feature selection consistency diagnostic",
        "Added dataset missingness and technical artifact audit"
    ]
}
with open(os.path.join(DRIVE_RESULTS_PATH, "experiment_config.json"), "w") as f:
    json.dump(config, f, indent=4)

print("\nSaved files:")
for f in sorted(os.listdir(DRIVE_RESULTS_PATH)):
    size = os.path.getsize(os.path.join(DRIVE_RESULTS_PATH, f))
    print(f"  {f:55s} {size/1024:.1f} KB")